<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/dermomodel_mobilenetv3L.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os
import shutil
import tensorflow as tf

from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    ReduceLROnPlateau,
    EarlyStopping
)

from tensorflow.keras import mixed_precision
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.applications.mobilenet_v3 import preprocess_input

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.test.gpu_device_name())

TensorFlow version: 2.20.0
GPU: /device:GPU:0


In [3]:
BASE = "/content/newdata"
IMG_SRC = "/content/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints"
MODEL_SAVE_PATH = "/content/drive/MyDrive/Colab Notebooks/Models/dermoscopy/mobilenetv4_middle_fusion.keras"

In [4]:
print(os.path.exists(IMG_SRC))

True


In [ ]:
def safe_copytree(src, dst):
    for root, dirs, files in os.walk(src):
        for file in files:
            src_file = os.path.join(root, file)
            rel_path = os.path.relpath(root, src)
            dst_dir = os.path.join(dst, rel_path)
            os.makedirs(dst_dir, exist_ok=True)

            try:
                shutil.copy2(src_file, os.path.join(dst_dir, file))
            except Exception as e:
                print("Skipped:", src_file, "->", e)

safe_copytree(IMG_SRC, BASE)

In [5]:
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)

In [6]:
batch_size = 16
image_size = 256

In [7]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
])

In [8]:
def add_edge_map(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    gray = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)
    edge = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))
    edge = edge / (tf.reduce_max(edge) + 1e-6)
    edge = tf.cast(edge, tf.float32)

    return (image, edge), label

In [9]:
def prepare_dataset(path, shuffle):

    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(image_size, image_size),
        batch_size=batch_size,
        label_mode='categorical',
        shuffle=shuffle
    )

    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = prepare_dataset(f"{BASE}/train", True)
val_ds = prepare_dataset(f"{BASE}/valid", False)
test_ds = prepare_dataset(f"{BASE}/test", False)

Found 8012 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [10]:
def create_dual_model():

    rgb_input = layers.Input(
        shape=(image_size, image_size, 3),
        name="rgb_input"
    )

    x_rgb = data_augmentation(rgb_input)
    base = MobileNetV3Large(
        include_top=False,
        weights='imagenet',
        input_tensor=x_rgb
    )

    # Freeze most layers first
    base.trainable = True

    for layer in base.layers[:-30]:
        layer.trainable = False

    # -----------------------------------------------------
    # MIDDLE FEATURE MAP
    # -----------------------------------------------------

    middle_layer = base.get_layer("expanded_conv_6_project").output

    # -----------------------------------------------------
    # EDGE INPUT
    # -----------------------------------------------------

    edge_input = layers.Input(
        shape=(image_size, image_size, 1),
        name="edge_input"
    )

    x = layers.Conv2D(32,3,padding='same',activation='relu',dtype='float32')(edge_input)
    x = layers.BatchNormalization(dtype='float32')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv2D(64,3,padding='same',activation='relu',dtype='float32')(x)
    x = layers.BatchNormalization(dtype='float32')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Conv2D(128,3,padding='same',activation='relu',dtype='float32')(x)
    x = layers.BatchNormalization(dtype='float32')(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Dropout(0.3)(x)

    # -----------------------------------------------------
    # RESIZE EDGE FEATURES
    # -----------------------------------------------------

    target_size = middle_layer.shape[1:3]
    x = layers.Resizing(target_size[0],target_size[1])(x)

    # -----------------------------------------------------
    # FEATURE FUSION
    # -----------------------------------------------------

    fused = layers.Concatenate()([middle_layer, x])

    # -----------------------------------------------------
    # POST-FUSION CNN
    # -----------------------------------------------------

    fused = layers.Conv2D(256,3,padding='same',activation='relu',dtype='float32')(fused)
    fused = layers.BatchNormalization(dtype='float32')(fused)
    fused = layers.MaxPooling2D(2)(fused)
    fused = layers.Dropout(0.3)(fused)
    fused = layers.Conv2D(512,3,padding='same',activation='relu',dtype='float32')(fused)
    fused = layers.BatchNormalization(dtype='float32')(fused)
    fused = layers.GlobalAveragePooling2D()(fused)
    fused = layers.Dropout(0.5)(fused)

    # -----------------------------------------------------
    # OUTPUT
    # -----------------------------------------------------

    outputs = layers.Dense(2,activation='softmax',dtype='float32')(fused)
    model = Model(inputs=[rgb_input, edge_input],outputs=outputs)

    # -----------------------------------------------------
    # COMPILE
    # -----------------------------------------------------

    model.compile(

        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(
            label_smoothing=0.05
        ),
        metrics=['accuracy']
    )

    return model

In [11]:
model = create_dual_model()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/applications/mobilenet_v3.py:519: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ rgb_input           │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 256, 256,  │          0 │ rgb_input[0][0]   │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 256, 256,  │          0 │ sequential[0][0]  │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, 128, 128,  │        432 │ rescaling[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, 128, 128,  │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 128, 128,  │          0 │ conv_bn[0][0]     │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 128,  │        144 │ activation[0][0]  │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 128, 128,  │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 128, 128,  │          0 │ expanded_conv_de… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 128, 128,  │        256 │ re_lu[0][0]       │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 128, 128,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_add   │ (None, 128, 128,  │          0 │ activation[0][0], │
│ (Add)               │ 16)               │            │ expanded_conv_pr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 128, 128,  │      1,024 │ expanded_conv_ad… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 128, 128,  │        256 │ expanded_conv_1_… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 128, 128,  │          0 │ expanded_conv_1_… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 129, 129,  │          0 │ re_lu_1[0][0]     │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 64, 64,    │        576 │ expanded_conv_1_

 Total params: 1,853,538 (7.07 MB)

 Trainable params: 1,755,330 (6.70 MB)

 Non-trainable params: 98,208 (383.62 KB)

In [12]:
checkpoint_best = ModelCheckpoint(
    filepath=f"{CHECKPOINT_DIR}/best_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-7
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)


In [13]:
print("\n==============================")
print("TRAINING")
print("==============================\n")

history = model.fit(train_ds,epochs=15,validation_data=val_ds,callbacks=[checkpoint_best,reduce_lr,early_stop])


TRAINING

Epoch 1/15
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 269ms/step - accuracy: 0.6942 - loss: 0.6044
Epoch 1: val_accuracy improved from None to 0.89211, saving model to /content/drive/MyDrive/checkpoints/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/checkpoints/best_model.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 200s 355ms/step - accuracy: 0.7796 - loss: 0.5159 - val_accuracy: 0.8921 - val_loss: 0.3318 - learning_rate: 1.0000e-04
Epoch 2/15
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step - accuracy: 0.8780 - loss: 0.3696
Epoch 2: val_accuracy improved from 0.89211 to 0.89510, saving model to /content/drive/MyDrive/checkpoints/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/checkpoints/best_model.keras
501/501 ━━━━━━━━━━━━━━━━━━━━ 149s 296ms/step - accuracy: 0.8817 - loss: 0.3582 - val_accuracy: 0.8951 - val_loss: 0.3426 - learning_rate: 1.0000e-04
Epoch 3/15
501/501 ━━━━━━━━━━━━━━━━━━━━ 0s 249ms/step - accuracy: 0.8869 - loss: 0.3356
Epoch 3:

In [14]:
print("\n==============================")
print("TEST EVALUATION")
print("==============================\n")

loss, acc = model.evaluate(test_ds)

print(f"\nFinal Test Accuracy: {acc:.4f}")



TEST EVALUATION

63/63 ━━━━━━━━━━━━━━━━━━━━ 24s 375ms/step - accuracy: 0.8802 - loss: 0.3449

Final Test Accuracy: 0.8802


In [15]:
model.save(MODEL_SAVE_PATH)

print("\nModel Saved Successfully!")


Model Saved Successfully!
